In [2]:

#!/usr/bin/env python
# coding: utf-8

# ============================================================
# EGH490 VIRTUAL POWER PLANT DEMONSTRATOR
# ============================================================
#
# Main simulation and hardware-demonstration script.
#
# The simulated VPP consists of five residential premises
# containing different combinations of:
#   - household electrical demand,
#   - rooftop solar PV, and
#   - battery energy storage.
#
# The hospital load and wind/DFIG model are external
# demonstration elements and are not considered VPP assets.
#
# Simulation resolution: 30 minutes.
# The profile data repeats continuously during hardware
# demonstrations.
# ============================================================


# ============================================================
# IMPORTS AND PROJECT CONFIGURATION
# ============================================================

import os
import sys
import time

# Absolute project root on the Jetson.
# Adding this directory to sys.path allows project modules,
# such as the hardware interface, to be imported when the
# notebook/script is launched from another working directory.

PROJECT_ROOT = "/home/jpobar/egh490_vpp_project"

# Make project folders importable
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

# Startup information used to confirm that the expected
# project directory and Python import path are being used.
print("Working directory:", os.getcwd())
print("Project root added:", PROJECT_ROOT)

import numpy as np
import pandas as pd
from datetime import datetime

from hardware.hardware_interface import HardwareInterface
print("HardwareInterface imported successfully")

# ============================================================
# PROFILE DATA
# ============================================================
# Load the prepared seasonal datasets containing the
# 30-minute household demand and PV generation profiles.
#
# Both datasets are loaded here so the simulation can later
# support seasonal scenario selection. The current scenario
# below uses the summer dataset.


summer_data = pd.read_csv(
    "SUMMER Generation and Load profiles.csv"
)

winter_data = pd.read_csv(
    "WINTER Generation and Load profiles.csv"
)

# Basic validation of the selected profile dataset.
# These checks help identify missing data or unexpected CSV
# structure before simulation objects are created.

print("\n=== SUMMER PROFILE DATA CHECK ===")
print(f"Rows loaded: {len(summer_data)}")
print(f"Columns loaded: {len(summer_data.columns)}")
print(f"Missing values: {summer_data.isna().sum().sum()}")

print("\nFirst 5 rows:")
print(summer_data.head())

print("\nColumn names:")
print(summer_data.columns.tolist())


Working directory: /home/jpobar/egh490_vpp_project/Notebooks
Project root added: /home/jpobar/egh490_vpp_project
HardwareInterface imported successfully

=== SUMMER PROFILE DATA CHECK ===
Rows loaded: 336
Columns loaded: 10
Missing values: 0

First 5 rows:
  LOAD_SOURCE_TIMESTEP PV_SOURCE_TIMESTAMP  PV_6.6kW_kW  PV_13.3kW_kW  \
0      04/02/2013 0:00     07/01/2013 0:00          0.0           0.0   
1      04/02/2013 0:30     07/01/2013 0:30          0.0           0.0   
2      04/02/2013 1:00     07/01/2013 1:00          0.0           0.0   
3      04/02/2013 1:30     07/01/2013 1:30          0.0           0.0   
4      04/02/2013 2:00     07/01/2013 2:00          0.0           0.0   

   PV_20kW_kW  H1_HI_B_kW  H2_MED_B_kW  H3_LOW_A_Kw  H4_Hi_A_kW  H5_Med_A_kW  
0         0.0       0.706        0.150        0.135       0.213        0.353  
1         0.0       0.695        0.154        0.136       0.216        0.390  
2         0.0       0.646        0.177        0.059       0.202    

Define simulation Objects


In [3]:
# ============================================================
# VPP SIMULATION ORCHESTRATOR
# ============================================================


class VPPSimulation:
    """
    Coordinates the VPP simulation at each discrete timestep.

    The simulation object does not decide how DER should operate.
    Instead, it:

        1. selects the current load/PV profile values,
        2. updates connection constraints,
        3. gathers the pre-control state of each house,
        4. requests control signals from the VPP controller,
        5. executes each house and battery response,
        6. calculates VPP and wider-network power balances,
        7. stores results for analysis and hardware display.

    The VPP boundary contains the five participating houses only.
    Hospital and wind resources are external demonstration elements.
    """

    
    def __init__(self,
                 houses,
                 hospital,
                 wind,
                 controller,
                 timestep_hours,
                 num_steps
                ):
        
        # Participating residential VPP assets.
        self.houses = houses

        # External demonstration resources.
        # These are included in the wider network power balance,
        # but are not internal VPP assets.
        self.hospital = hospital
        self.wind = wind

        # Supervisory controller responsible for selecting the
        # VPP operating mode and issuing house-level targets.
        self.controller = controller

        # Duration of one logical simulation timestep and number
        # of available profile samples.
        self.timestep_hours = timestep_hours
        self.num_steps = num_steps

        # Absolute simulation step. This continues increasing even
        # when the finite profile dataset loops back to its start.
        self.current_step = 0

        # VPP-level results recorded once per simulation timestep.
        self.simulation_results_summary = []

        # Separate timestep results retained for each house.
        self.house_results_log = {
            house.name: [] for house in houses
        }

        
    def step(self, external_inputs=None):

        """
        Execute one logical VPP simulation timestep.

        external_inputs contains supervisory commands originating
        outside the normal controller logic, such as simulated or
        physical network-support and PV-sharing inputs.
        """
        
        
        if external_inputs is None:
            
            external_inputs = {}

        # Loop the finite load/PV dataset continuously.
        # current_step continues increasing while profile_index
        # wraps back to zero after the final profile sample.
            
        profile_index = self.current_step % self.num_steps

        # Convert the absolute simulation step into time-of-day.
        # For a 0.5 h timestep:
        #   step 0  = 00:00
        #   step 1  = 00:30
        #   step 32 = 16:00
        
        hour_of_day = (
            self.current_step * self.timestep_hours
        ) % 24

        
        # --------------------------------------------------------
        # UPDATE CONNECTION EXPORT LIMITS
        # --------------------------------------------------------
        #
        # A10:
        # During an explicit network-support event, the
        # demonstrator assumes that additional connection export
        # capacity may temporarily be made available up to the
        # modelled maximum connection capability.
        #
        # Export limits are updated BEFORE house states are passed
        # to the controller so that reported VPP discharge
        # capability reflects the applicable connection limit.

        network_support_request = external_inputs.get(
            "network_support_request",
            False
        )
        
        for house in self.houses:
            house.update_export_limit(network_support_request)
        
        # --------------------------------------------------------
        # GATHER PRE-CONTROL SYSTEM STATE
        # --------------------------------------------------------

        # Each house reports its electrical state and available
        # DER flexibility before any control action is applied.

        house_states = [
            house.get_state(
                profile_index,
                self.timestep_hours
                )
                for house in self.houses
        ]
        
        # Evaluate external demonstration resources separately.
        # These resources are not included in the VPP boundary.   
        hospital_result = self.hospital.step(profile_index)
        wind_result = self.wind.step(profile_index)

    
        # --------------------------------------------------------
        # VPP SUPERVISORY CONTROL
        # --------------------------------------------------------

        # Determine the operating mode and any house-level
        # dispatch or shared-PV targets for this timestep.
        signals = self.controller.get_signals(
            hour_of_day,
            external_inputs,
            house_states,
        )


        # --------------------------------------------------------
        # EXECUTE HOUSE AND DER RESPONSE
        # --------------------------------------------------------

        # Each house applies the controller signals while its
        # House and Battery objects enforce local physical limits.

        house_results = [
            house.step(
                profile_index,
                self.timestep_hours,
                signals
            )
            for house in self.houses
        ]

        # --------------------------------------------------------
        # AGGREGATE PARTICIPATING HOUSE RESULTS
        # --------------------------------------------------------

        total_house_demand_kw = sum(
            h["load_kw"]
            for h in house_results
        )

        total_house_generation_kw = sum(
            h["generation_kw"]
            for h in house_results
        )

        total_house_net_power_kw = sum(
            h["net_power_kw"]
            for h in house_results
        )

        total_house_net_power_before_battery_kw = sum(
            h["net_power_kw_before_battery"]
            for h in house_results
        )

        # Positive battery power represents discharge.
        # Negative battery power represents charging.
        total_battery_power_kw = sum(
            h["battery_power_kw"]
            for h in house_results
        )

        # Aggregate stored battery energy across all
        # battery-equipped VPP participants.
        total_battery_soc_kwh = sum(
            h["battery_soc_kwh"]
            for h in house_results
        )

        total_battery_discharge_kw = sum(
            h["battery_discharge_kw"]
            for h in house_results
        )

        total_battery_charge_kw = sum(
            h["battery_charge_kw"]
            for h in house_results
        )

        # External demonstration quantities.
        total_local_load_kw = (
            hospital_result["load_kw"]
        )

        total_wind_generation_kw = (
            wind_result["generation_kw"]
        )

        # --------------------------------------------------------
        # VPP GRID EXCHANGE
        # --------------------------------------------------------
        #
        # Only the five participating houses belong to the VPP.
        #
        # Sign convention:
        #   positive VPP net power = net grid import
        #   negative VPP net power = net grid export

        
        vpp_net_power_kw = total_house_net_power_kw
        
        vpp_grid_import_kw = max(
            vpp_net_power_kw,
            0.0
        )
        
        vpp_grid_export_kw = -min(
            vpp_net_power_kw,
            0.0
        )
        
        
        # --------------------------------------------------------
        # WIDER DEMONSTRATION / NETWORK POWER BALANCE
        # --------------------------------------------------------
        #
        # Hospital and wind/DFIG are included here only to show
        # the interaction between the VPP and external resources.
        # They do not alter the defined VPP asset boundary.
        
        network_net_power_kw = (
            vpp_net_power_kw
            + total_local_load_kw
            - total_wind_generation_kw
        )
        
        network_grid_import_kw = max(
            network_net_power_kw,
            0.0
        )
        
        network_grid_export_kw = -min(
            network_net_power_kw,
            0.0
        )
        
        
        # --------------------------------------------------------
        # SIMULATION TIME
        # --------------------------------------------------------

        # Absolute elapsed simulation time does not reset when
        # the underlying profile dataset loops.

        elapsed_hours = self.current_step * self.timestep_hours
        day_number = int(elapsed_hours // 24) + 1
        


        # --------------------------------------------------------
        # BUILD TIMESTEP RESULT
        # --------------------------------------------------------
        #
        # Store the quantities required for:
        #   - technical analysis,
        #   - later economic analysis,
        #   - dashboard visualisation,
        #   - debugging and result export.

        timestep_result = {
            "Simulation_step": self.current_step,
            "Profile_index": profile_index,
            "Elapsed_hours": elapsed_hours,
            "Day_number": day_number,
            "Hour_of_day": hour_of_day,

            "House_demand_kw": total_house_demand_kw,
            "House_generation_kw": total_house_generation_kw,
            "House_net_power_before_battery_kw":
                total_house_net_power_before_battery_kw,

            "Battery_power_kw": total_battery_power_kw,
            "Battery_soc_kwh": total_battery_soc_kwh,
            "Battery_discharge_kw": total_battery_discharge_kw,
            "Battery_charge_kw": total_battery_charge_kw,
            
            "Battery_discharge_kwh": total_battery_discharge_kw * self.timestep_hours,

            "Battery_charge_kwh": total_battery_charge_kw * self.timestep_hours,

            # VPP power balance
            "VPP_net_power_kw": vpp_net_power_kw,
            "VPP_grid_import_kw": vpp_grid_import_kw,
            "VPP_grid_export_kw": vpp_grid_export_kw,
            
            # External demonstration resources
            "Hospital_load_kw": total_local_load_kw,
            "Wind_generation_kw": total_wind_generation_kw,
            
            # Wider demonstration/network power balance
            "Network_net_power_kw": network_net_power_kw,
            "Network_grid_import_kw": network_grid_import_kw,
            "Network_grid_export_kw": network_grid_export_kw,

            "VPP_mode": signals["mode"],
            "is_peak_period": signals["is_peak_period"],
            "Is_event_period": signals["is_event_period"],
            "Is_cheap_energy_period":signals["is_cheap_energy_period"],
            "PV_sharing_enabled": signals["pv_sharing_enabled"],

            "Requested_support_kw": signals["requested_support_kw"],
            "Available_VPP_discharge_kw":signals["total_available_discharge_kw"],
            
            "Shared_PV_available_kw": signals["total_shared_pv_available_kw"],
            "Shared_PV_transferred_kw": signals["total_shared_pv_transferred_kw"],
            
            "house_shared_pv_targets": signals["house_shared_pv_targets"],
            "House_shared_export_targets": signals["house_shared_export_targets"],
            "VPP_available_charge_kw": {
                house.name: state["vpp_available_charge_kw"]
                for house, state in zip(
                    self.houses,
                    house_states
                )
            },
        }


        # Store the VPP-level result for this timestep.
        self.simulation_results_summary.append(timestep_result)


        # Store detailed individual-house results separately.
        for result in house_results:
            house_name = result["name"]
            self.house_results_log[house_name].append(result)


        # Advance the absolute simulation clock.
        # Profile wrapping occurs at the beginning of the next
        # call through the modulo operation on profile_index.
        
        self.current_step += 1


        # Return the latest result
        return timestep_result


# ============================================================
# RESIDENTIAL HOUSE / CONNECTION-POINT MODEL
# ============================================================

class House:
    """
    Represents one participating residential premises in the VPP.

    Each house may contain:
        - a household load profile,
        - a rooftop PV generation profile,
        - an optional battery energy storage system,
        - PV inverter metadata, and
        - a simplified grid connection/export constraint.

    The House object is responsible for enforcing local physical
    constraints after receiving supervisory targets from the
    VPP controller.

    Sign convention at the connection point:
        positive net power = grid import
        negative net power = grid export
    """

    def __init__(
        self,
        name,
        load_profile,
        generation_profile,
        battery=None,
        pv_array_kwp=0.0,
        pv_inverter_limit_kw=0.0,
        connection_type="single_phase",
        connection_phases=1,
        min_export_kw_per_phase=1.5,
        max_export_kw_per_phase=10.0
    ):

        self.name = name

        # Time-series profiles used by this premises.
        self.load_profile = load_profile
        self.generation_profile = generation_profile

        # Optional Battery object. A value of None represents
        # a participating house without battery storage.
        self.battery = battery

        # ----------------------------------------------------
        # PV SYSTEM METADATA
        # ----------------------------------------------------
        #
        # PV generation is supplied by the prepared SAM profile.
        # These ratings describe the installed system and are
        # retained for reporting rather than re-limiting the
        # imported AC generation profile here.
        self.pv_array_kwp = pv_array_kwp
        self.pv_inverter_limit_kw = pv_inverter_limit_kw

        # ----------------------------------------------------
        # GRID CONNECTION
        # ----------------------------------------------------

        self.connection_type = connection_type
        self.connection_phases = connection_phases

        self.min_export_kw_per_phase = (
            min_export_kw_per_phase
        )

        self.max_export_kw_per_phase = (
            max_export_kw_per_phase
        )

        # The simplified simulation uses aggregate connection
        # limits rather than modelling individual phase flows.
        self.min_export_limit_kw = (
            min_export_kw_per_phase
            * connection_phases
        )

        self.max_export_limit_kw = (
            max_export_kw_per_phase
            * connection_phases
        )

        # Normal dynamic export allowance applicable to the
        # premises. This can later be replaced by a time-varying
        # network/DNSP input.
        self.dynamic_export_limit_kw = (
            self.max_export_limit_kw
        )

        # Export limit currently applied to the premises.
        #
        # Under normal operation this equals the dynamic export
        # limit. A10 may temporarily increase it during an
        # explicit network-support event.
        self.current_export_limit_kw = (
            self.dynamic_export_limit_kw
        )
            
        
    def get_state(
        self,
        profile_index,
        timestep_hours
    ):
        """
        Determine the pre-control electrical state and available
        DER flexibility of this premises.

        This method does not dispatch the battery. It reports the
        quantities required by the VPP controller to make its
        supervisory decision for the current timestep.
        """

        # Current household demand and available PV generation
        # from the selected profile sample.
        load_kw = self.load_profile[profile_index]
        generation_kw = (
            self.generation_profile[profile_index]
        )

        # Connection-point power before any battery action.
        #
        # Positive = household deficit / potential grid import.
        # Negative = PV surplus / potential grid export.
        net_power_kw_before_battery = (
            load_kw - generation_kw
        )

        # Household demand that remains after using its own PV.
        local_import_kw = max(
            net_power_kw_before_battery,
            0.0
        )

        # PV remaining after satisfying this house's own load.
        local_pv_surplus_kw = max(
            -net_power_kw_before_battery,
            0.0
        )
        # A house without a battery can still receive shared PV
        # to offset its current household deficit.
        #
        # For battery-equipped houses this value is expanded
        # below to include remaining battery charging capability.
        shared_pv_receiving_capacity_kw = (
            local_import_kw
        )
        
        # Default values for houses without battery storage.
        battery_soc_kwh = None
        battery_soc_percent = None

        max_discharge_kw = 0.0
        max_charge_kw = 0.0

        available_discharge_kw = 0.0
        available_charge_kw = 0.0

        export_headroom_kw = 0.0
        vpp_available_discharge_kw = 0.0

        local_battery_charge_from_pv_kw = 0.0

        # Without a battery, all PV remaining after local demand
        # is potentially available for VPP sharing.
        shared_pv_available_kw = local_pv_surplus_kw

        vpp_available_charge_kw = 0.0
        
        

        if self.battery is not None:

            # Current physical battery state.
            battery_soc_kwh = (
                self.battery.get_soc_kwh()
            )

            battery_soc_percent = (
                self.battery.get_soc_percent()
            )

            max_discharge_kw = (
                self.battery.max_discharge_kw
            )

            max_charge_kw = (
                self.battery.max_charge_kw
            )

            # ------------------------------------------------
            # PHYSICAL BATTERY DISCHARGE CAPABILITY
            # ------------------------------------------------

            # Energy physically available above the battery's
            # minimum usable SOC.
            #
            # The future VPP operating reserve (e.g. 20% SOC)
            # will be a controller-level requirement and is
            # intentionally separate from this physical limit.
            available_energy_kwh = max(
                0.0,
                self.battery.soc_kwh
                - self.battery.min_soc_kwh
            )

            # Maximum discharge that can be sustained for the
            # complete simulation timestep, subject to both
            # stored energy and battery power capability.
            available_discharge_kw = min(
                self.battery.max_discharge_kw,
                available_energy_kwh
                * self.battery.efficiency
                / timestep_hours
            )

            # --------------------------------------------------
            # CONNECTION-CONSTRAINED VPP DISCHARGE CAPABILITY
            # ------------------------------------------------

            # PV export that would already exist without battery
            # discharge.
            export_before_battery_kw = max(
                -net_power_kw_before_battery,
                0.0
            )

            # Remaining export capacity under the CURRENT
            # connection limit.
            #
            # This deliberately does not use the A09 shared-PV
            # allowance. Any additional A09 export capacity is
            # reserved specifically for allocated shared PV and
            # must not create extra battery export capability.
            export_headroom_kw = max(
                self.current_export_limit_kw
                - export_before_battery_kw,
                0.0
            )

            # Battery discharge can:
            #   1. offset this house's existing grid import, then
            #   2. export through remaining connection headroom.
            connection_permitted_discharge_kw = (
                local_import_kw
                + export_headroom_kw
            )

            # VPP-visible discharge capability is therefore the
            # smaller of the physical battery capability and the
            # connection-permitted capability.
            vpp_available_discharge_kw = min(
                available_discharge_kw,
                connection_permitted_discharge_kw
            )
            
            # ------------------------------------------------
            # BATTERY CHARGING CAPABILITY
            # ------------------------------------------------

            # Remaining physical storage space.
            available_space_kwh = max(
                0.0,
                self.battery.capacity_kwh
                - self.battery.soc_kwh
            )

            # Maximum charging power that can be accepted for the
            # complete timestep without exceeding battery capacity.
            available_charge_kw = min(
                self.battery.max_charge_kw,
                available_space_kwh / (
                    timestep_hours
                    * self.battery.efficiency
                )
            )

            # Own surplus PV is allocated to the house's own
            # battery before that PV becomes available for
            # sharing with other VPP participants.
            local_battery_charge_from_pv_kw = min(
                local_pv_surplus_kw,
                available_charge_kw
            )

            # Genuine PV surplus remaining after:
            #   own household demand, then
            #   own battery charging capability.
            shared_pv_available_kw = max(
                local_pv_surplus_kw
                - local_battery_charge_from_pv_kw,
                0.0
            )

            # Battery charging capability remaining after allowing
            # for this house's own surplus PV.
            vpp_available_charge_kw = max(
                available_charge_kw
                - local_battery_charge_from_pv_kw,
                0.0
            )

            # Shared PV received from another participant can first
            # offset this house's unmet demand and can then use any
            # remaining battery charging capability.
            shared_pv_receiving_capacity_kw = (
                local_import_kw
                + vpp_available_charge_kw
            )

        return {
            # Identity and pre-control electrical state
            "name": self.name,
            "load_kw": load_kw,
            "generation_kw": generation_kw,
            "net_power_kw_before_battery":
                net_power_kw_before_battery,
            "local_import_kw": local_import_kw,

            # Battery presence and state
            "has_battery": self.battery is not None,
            "battery_soc_kwh": battery_soc_kwh,
            "battery_soc_percent": battery_soc_percent,

            # Physical battery capability
            "max_discharge_kw": max_discharge_kw,
            "max_charge_kw": max_charge_kw,
            "available_discharge_kw":
                available_discharge_kw,
            "available_charge_kw":
                available_charge_kw,

            # PV sharing capability
            "local_pv_surplus_kw":
                local_pv_surplus_kw,
            "local_battery_charge_from_pv_kw":
                local_battery_charge_from_pv_kw,
            "shared_pv_available_kw":
                shared_pv_available_kw,
            "vpp_available_charge_kw":
                vpp_available_charge_kw,
            "shared_pv_receiving_capacity_kw":
                shared_pv_receiving_capacity_kw,

            # Connection-constrained VPP capability
            "export_headroom_kw": export_headroom_kw,
            "vpp_available_discharge_kw":
                vpp_available_discharge_kw,

            # Installed PV and connection metadata
            "pv_array_kwp": self.pv_array_kwp,
            "pv_inverter_limit_kw":
                self.pv_inverter_limit_kw,

            "connection_type": self.connection_type,
            "connection_phases":
                self.connection_phases,

            "min_export_limit_kw":
                self.min_export_limit_kw,
            "max_export_limit_kw":
                self.max_export_limit_kw,

            "dynamic_export_limit_kw":
                self.dynamic_export_limit_kw,
            "current_export_limit_kw":
                self.current_export_limit_kw,
        }
            
    def update_export_limit(
        self,
        network_support_request=False
    ):
        """
        Update the connection export limit applied to this house.
    
        Under normal operation, the premises is constrained by its
        current dynamic export limit.
    
        A10:
        During an explicit network-support event, the demonstrator
        assumes that additional network export capacity has been made
        available, allowing the premises to operate up to its modelled
        maximum connection export capability.
    
        This is a demonstrator scenario assumption and is separate
        from the A09 shared-PV-specific export allowance.
        """
    
        if network_support_request:
            # Temporary A10 network-support allowance.
            self.current_export_limit_kw = (
                self.max_export_limit_kw
            )
    
        else:
            # Restore normal dynamic connection constraint.
            self.current_export_limit_kw = (
                self.dynamic_export_limit_kw
            )
  

        

    def step(
        self,
        t,
        timestep_hours,
        signals,
    ):
        """
        Execute one physical house timestep using the supervisory
        signals supplied by the VPP controller.

        The controller requests an operating behaviour, while this
        method applies that behaviour subject to the house's battery,
        PV and connection-point constraints.

        Battery power sign convention:
            positive = battery discharge
            negative = battery charge

        Connection-point sign convention:
            positive = grid import
            negative = grid export
        """

        # ----------------------------------------------------
        # CURRENT HOUSE ELECTRICAL STATE
        # ----------------------------------------------------

        # Retrieve household demand and available PV generation
        # for the current profile timestep.
        load_kw = self.load_profile[t]
        generation_kw = self.generation_profile[t]

        # Net premises power before battery action.
        #
        # Positive = household demand exceeds PV generation.
        # Negative = PV generation exceeds household demand.
        net_power_kw_bb = (
            load_kw - generation_kw
        )
        
        # ----------------------------------------------------
        # CONNECTION-POINT EXPORT LIMIT
        # ----------------------------------------------------

        # Current connection export limit applicable to normal
        # house operation.
        #
        # This normally equals the dynamic export limit, but may
        # already have been increased by A10 if an explicit
        # network-support event is active.
        current_export_limit_kw = (
            self.current_export_limit_kw
        )


        # ----------------------------------------------------
        # A09 - SHARED-PV-SPECIFIC EXPORT ALLOWANCE
        # ----------------------------------------------------

        # The controller may allocate genuine surplus PV from this
        # house to other participating VPP houses.
        shared_pv_export_target_kw = signals[
            "house_shared_export_targets"
        ].get(
            self.name,
            0.0
        )

        # A09:
        # If the allocated shared-PV export exceeds the normal
        # connection export allowance, the effective limit may be
        # temporarily increased only enough to accommodate that
        # shared-PV target.
        #
        # This additional allowance:
        #   - applies only when shared PV has been allocated,
        #   - cannot exceed the modelled maximum connection limit,
        #   - does NOT create additional battery export headroom.
        
        effective_export_limit_kw = min(
            max(
                current_export_limit_kw,
                shared_pv_export_target_kw
            ),
            self.max_export_limit_kw
        )
        
        # ----------------------------------------------------
        # NORMAL BATTERY EXPORT HEADROOM
        # ----------------------------------------------------

        # PV export that would already occur before battery action.
        export_before_battery_kw = max(
            -net_power_kw_bb,
            0.0
        )

        # Remaining connection capacity available for battery
        # export.
        #
        # Deliberately use current_export_limit_kw rather than
        # effective_export_limit_kw here. The additional A09
        # allowance is reserved for shared PV and must not increase
        # battery export capability.
        export_headroom_kw = max(
            current_export_limit_kw
            - export_before_battery_kw,
            0.0
        )

        # Quantities used later to record any restriction imposed
        # by the connection limit.
        battery_discharge_reduction_kw = 0.0
        pv_curtailment_kw = 0.0
        export_limit_active = False

        
        # ----------------------------------------------------
        # INITIALISE TIMESTEP BATTERY STATE
        # ----------------------------------------------------

        # Battery power remains zero for houses without storage
        # or whenever the active controller mode requests no
        # battery action.
        battery_power_kw = 0.0

        # Houses without batteries report zero SOC in the physical
        # timestep results used by the demonstrator and hardware
        # display.
        battery_soc_kwh = 0.0
        battery_soc_percent = 0.0
        
        
        # ----------------------------------------------------
        # BATTERY DISPATCH
        # ----------------------------------------------------

        
        if self.battery is not None:

            # Read the supervisory operating mode and supporting
            # controller flags for this timestep.
            mode = signals["mode"]

            is_peak_period = (
                signals["is_peak_period"]
            )

            is_grid_peak_limit_exceeded = (
                signals["is_grid_peak_limit_exceeded"]
            )


            is_cheap_energy_period = (
                signals["is_cheap_energy_period"]
            )

            
            
            # ------------------------------------------------
            # RECEIVED SHARED PV
            # ------------------------------------------------

            # Shared PV allocated to this house can first offset
            # its remaining household demand and can then create
            # surplus power available for battery charging.
            shared_pv_allocation_kw = signals[
                "house_shared_pv_targets"
            ].get(
                self.name,
                0.0
            )

            # Virtual net power used for battery dispatch after
            # accounting for the VPP-directed shared-PV allocation.
            #
            # The actual physical connection-point power is
            # calculated separately later in this method.
            effective_net_power_kw = (
                net_power_kw_bb
                - shared_pv_allocation_kw
            )


            # ------------------------------------------------
            # MODE: SELF CONSUMPTION
            # ------------------------------------------------
            #
            # Normal DER operation:
            #   - surplus energy charges the battery,
            #   - remaining household deficit is supplied by
            #     battery discharge where possible.
            if mode == "self_consumption":

                if effective_net_power_kw < 0:
                    # charge from remaining surplus
                    battery_power_kw = self.battery.charge(-effective_net_power_kw, timestep_hours)

            
                elif effective_net_power_kw > 0:
                    # discharge for remaining deficit
                    battery_power_kw = self.battery.discharge(effective_net_power_kw, timestep_hours)
                
                else: 
                    battery_power_kw = 0.0
            
            # ------------------------------------------------
            # MODE: BATTERY HOLD
            # ------------------------------------------------
            #
            # Preserve stored battery energy for a future
            # controller objective or higher-value period.
            elif mode == "battery_hold":
                battery_power_kw = 0.0

            # ------------------------------------------------
            # MODE: PEAK SHAVING
            # ------------------------------------------------
            #
            # During an enabled peak period or when the configured
            # VPP import threshold is exceeded, battery discharge
            # is used to reduce local demand.
            #
            # This mode does not intentionally discharge additional
            # battery energy for grid export.      
                
            elif mode == "peak_shaving":
                
                # Continue to absorb available surplus energy.
                if effective_net_power_kw < 0:
                    battery_power_kw = self.battery.charge(
                        -effective_net_power_kw,
                        timestep_hours
                    )
                # Discharge only enough to offset the remaining
                # local deficit during the applicable peak condition.
                elif (
                    effective_net_power_kw > 0
                    and (
                        is_peak_period
                        or is_grid_peak_limit_exceeded
                    )
                ):
                    battery_power_kw = self.battery.discharge(
                        effective_net_power_kw,
                        timestep_hours
                    )
            
                else:
                    battery_power_kw = 0.0

            # ------------------------------------------------
            # MODE: ECONOMIC EXPORT
            # ------------------------------------------------
            #
            # During a designated high-value export period,
            # deliberately discharge the battery beyond local
            # household demand so that available stored energy
            # may be exported to the grid.
            #
            # Unlike network-support operation, this mode does
            # not invoke A10. Battery export therefore remains
            # subject to the normal/current connection export
            # limit.
            
            elif mode == "economic_export":
            
                # Maximum discharge that can be accommodated
                # at the premises connection point consists of:
                #
                #   remaining local household demand
                #               +
                #   available grid-export headroom
            
                local_import_kw = max(
                    effective_net_power_kw,
                    0.0
                )
            
                maximum_connection_permitted_discharge_kw = (
                    local_import_kw
                    + export_headroom_kw
                )
            
                # Request the maximum economically available
                # battery discharge permitted by the connection.
                #
                # The Battery object subsequently enforces its
                # own physical power and stored-energy limits.
                battery_power_kw = self.battery.discharge(
                    maximum_connection_permitted_discharge_kw,
                    timestep_hours
                )
                    
            # ------------------------------------------------
            # MODE: NETWORK SUPPORT
            # ------------------------------------------------
            #
            # Respond to an explicit external network-support
            # request using the house-level discharge target
            # allocated by the VPP controller.
            #
            # Battery discharge first offsets this house's own
            # grid import. Any remaining requested discharge
            # becomes grid export and must remain within the
            # applicable connection export capability.
         
            elif mode == "network_support":

                requested_target_kw = signals[
                    "house_dispatch_targets"
                ].get(
                    self.name,
                    0.0
                )
            
                # Local demand that can be supplied without
                # creating grid export.
                local_import_kw = max(
                    effective_net_power_kw,
                    0.0
                )

                
                # Maximum battery discharge permitted at the
                # connection point:
                #
                #   local demand that can be offset
                #               +
                #   remaining battery export headroom
                
                maximum_connection_permitted_discharge_kw = (
                    local_import_kw
                    + export_headroom_kw
                )
                
                # Reduce the controller target if the connection
                # cannot accommodate the requested discharge.           
                
                permitted_target_kw = min(
                    requested_target_kw,
                    maximum_connection_permitted_discharge_kw
                )

                # Record any requested support that could not be
                # delivered because of the connection constraint.
                
                battery_discharge_reduction_kw = max(
                    requested_target_kw
                    - permitted_target_kw,
                    0.0
                )
                
                # Battery object applies its own remaining physical
                # power and stored-energy constraints.
                
                battery_power_kw = self.battery.discharge(
                    permitted_target_kw,
                    timestep_hours
                )

            # ------------------------------------------------
            # MODE: ARBITRAGE CHARGE
            # ------------------------------------------------
            #
            # Current placeholder for future economic operation.
            # During a designated cheap-energy period, request
            # maximum battery charging power.
            #
            # A later economic controller may replace this simple
            # rule with tariff- and forecast-based scheduling.
            
            elif mode == "arbitrage_charge":
                if is_cheap_energy_period:
                    battery_power_kw = self.battery.charge(self.battery.max_charge_kw, timestep_hours)

            
                
            # Unknown or unsupported modes default to no battery
            # action rather than creating an unintended dispatch.               
            else: 
                battery_power_kw = 0.0
            
            # Record battery SOC after the requested physical
            # battery action has been applied.
            
            battery_soc_kwh = self.battery.get_soc_kwh()
            battery_soc_percent = self.battery.get_soc_percent()
                   
        # ----------------------------------------------------
        # ACTUAL BATTERY POWER COMPONENTS
        # ----------------------------------------------------
        #
        # Convert the signed battery-power convention into
        # separate positive charge and discharge quantities for
        # logging, dashboard display and later energy accounting.

        battery_discharge_kw = max(battery_power_kw, 0.0)
        battery_charge_kw = -min(battery_power_kw, 0.0)

        # ----------------------------------------------------
        # POTENTIAL CONNECTION-POINT POWER
        # ----------------------------------------------------
        #
        # Calculate the premises power flow after the battery
        # action but before enforcing the final export constraint.
        #
        # Sign convention:
        #   positive = grid import
        #   negative = grid export

        potential_net_power_kw = (
            load_kw
            - generation_kw
            - battery_power_kw
        )

        # Convert negative net power into a positive export
        # magnitude for comparison with the export limit.
        potential_export_kw = max(
            -potential_net_power_kw,
            0.0
        )

        
        # ----------------------------------------------------
        # ENFORCE CONNECTION EXPORT LIMIT
        # ----------------------------------------------------
        #
        # Compare the potential premises export against the
        # effective export limit applicable to this timestep.
        #
        # The effective limit includes:
        #   - the normal/current connection limit, and
        #   - any A09 shared-PV-specific allowance.
        #
        # Battery discharge has already been constrained using
        # the normal/current connection limit, so any remaining
        # excess export at this stage is attributed to PV.

        if potential_export_kw > effective_export_limit_kw:

            export_limit_active = True

            excess_export_kw = (
                potential_export_kw
                - effective_export_limit_kw
            )

            # Curtail only the amount of PV required to bring the
            # premises export back within the applicable limit.
            #
            # Curtailment cannot exceed the PV generation that was
            # physically available during this timestep.
            pv_curtailment_kw = min(
                excess_export_kw,
                generation_kw
            )

        else:

            # The export-limit flag also records cases where a
            # requested battery discharge was reduced earlier
            # because of connection-point constraints.
            export_limit_active = (
                battery_discharge_reduction_kw > 0
            )


        # ----------------------------------------------------
        # ACTUAL PV GENERATION
        # ----------------------------------------------------

        # Available PV generation represents what the PV profile
        # could have produced before network-driven curtailment.
        #
        # Actual generation is the PV remaining after any required
        # export-limit curtailment.
        actual_generation_kw = (
            generation_kw
            - pv_curtailment_kw
        )


        # ----------------------------------------------------
        # FINAL CONNECTION-POINT POWER
        # ----------------------------------------------------
        #
        # Recalculate the actual premises grid exchange using:
        #   - household demand,
        #   - actual PV generation after curtailment, and
        #   - actual battery power.
        #
        # Shared-PV allocations are not directly subtracted here.
        # They represent VPP coordination between participating
        # premises, while this value represents the physical
        # meter flow at this individual connection point.

        final_net_power_kw = (
            load_kw
            - actual_generation_kw
            - battery_power_kw
        )
        
        # ----------------------------------------------------
        # BATTERY CONFIGURATION METADATA
        # ----------------------------------------------------
        #
        # Store installed battery information with the timestep
        # result so house-level exports remain self-describing.

        if self.battery is not None:

            battery_manufacturer = (
                self.battery.manufacturer
            )

            battery_model = (
                self.battery.model
            )

            battery_module_count = (
                self.battery.module_count
            )

            battery_capacity_kwh = (
                self.battery.capacity_kwh
            )

            battery_max_charge_kw = (
                self.battery.max_charge_kw
            )

            battery_max_discharge_kw = (
                self.battery.max_discharge_kw
            )

        else:

            # Explicit zero/None values allow non-battery houses
            # to use the same result structure as battery houses.
            battery_manufacturer = None
            battery_model = None
            battery_module_count = 0

            battery_capacity_kwh = 0.0
            battery_max_charge_kw = 0.0
            battery_max_discharge_kw = 0.0


            
        # ----------------------------------------------------
        # BUILD HOUSE TIMESTEP RESULT
        # ----------------------------------------------------
        #
        # Return both static configuration information and the
        # electrical results required for VPP aggregation,
        # analysis, dashboard display and later economic modelling.

        return {
            "name": self.name,

            # ------------------------------------------------
            # Physical house configuration
            # ------------------------------------------------
            "pv_array_kwp": self.pv_array_kwp,
            "pv_inverter_limit_kw":
                self.pv_inverter_limit_kw,

            "connection_type":
                self.connection_type,
            "connection_phases":
                self.connection_phases,

            "min_export_limit_kw":
                self.min_export_limit_kw,
            "max_export_limit_kw":
                self.max_export_limit_kw,
            "dynamic_export_limit_kw":
                self.dynamic_export_limit_kw,

            # ------------------------------------------------
            # Battery configuration
            # ------------------------------------------------
            "battery_manufacturer":
                battery_manufacturer,
            "battery_model":
                battery_model,
            "battery_module_count":
                battery_module_count,
            "battery_capacity_kwh":
                battery_capacity_kwh,
            "battery_max_charge_kw":
                battery_max_charge_kw,
            "battery_max_discharge_kw":
                battery_max_discharge_kw,

            # ------------------------------------------------
            # Household demand and PV generation
            # ------------------------------------------------
            "load_kw": load_kw,

            # PV available from the source profile before
            # connection-driven curtailment.
            "available_generation_kw":
                generation_kw,

            # PV prevented from being generated/exported because
            # of the applicable connection export constraint.
            "pv_curtailment_kw":
                pv_curtailment_kw,

            # PV actually retained after curtailment.
            "generation_kw":
                actual_generation_kw,

            # ------------------------------------------------
            # Connection-point state
            # ------------------------------------------------

            # Potential meter power before final PV curtailment.
            "potential_net_power_kw":
                potential_net_power_kw,

            # Normal/current connection export allowance.
            "current_export_limit_kw":
                current_export_limit_kw,

            # A09 shared-PV export allocation for this source
            # premises and the resulting effective export limit.
            "shared_pv_export_target_kw":
                shared_pv_export_target_kw,

            "effective_export_limit_kw":
                effective_export_limit_kw,

            # Records whether connection constraints affected
            # battery dispatch or PV export during this timestep.
            "export_limit_active":
                export_limit_active,

            "battery_discharge_reduction_kw":
                battery_discharge_reduction_kw,

            # ------------------------------------------------
            # Battery and final power-flow results
            # ------------------------------------------------
            "net_power_kw_before_battery":
                net_power_kw_bb,

            # Signed battery power:
            #   positive = discharge
            #   negative = charge
            "battery_power_kw":
                battery_power_kw,

            "battery_soc_kwh":
                battery_soc_kwh,
            "battery_soc_percent":
                battery_soc_percent,

            # Final physical premises grid exchange:
            #   positive = grid import
            #   negative = grid export
            "net_power_kw":
                final_net_power_kw,

            # Separate positive quantities retained for easier
            # energy accounting and dashboard display.
            "battery_charge_kw":
                battery_charge_kw,
            "battery_discharge_kw":
                battery_discharge_kw,
        }

        
# ============================================================
# BATTERY ENERGY STORAGE MODEL
# ============================================================

class Battery:
    """
    Represents the physical battery energy storage system
    installed at a participating house.

    The Battery object enforces:
        - usable energy capacity,
        - charge power limit,
        - discharge power limit,
        - charge/discharge efficiency, and
        - physical minimum SOC.

    Higher-level VPP operating policies, such as a protected
    20% SOC reserve, belong to the VPP controller rather than
    this physical battery model.

    Battery power convention used by House:
        positive = discharge
        negative = charge
    """

    def __init__(
        self,
        capacity_kwh,
        initial_soc_kwh,
        max_charge_kw,
        max_discharge_kw,
        efficiency=0.95,
        min_soc_kwh=0.0,
        manufacturer=None,
        model=None,
        module_count=None
    ):

        # ----------------------------------------------------
        # ENERGY STORAGE STATE
        # ----------------------------------------------------

        # Total usable battery energy capacity represented by
        # the simulation.
        self.capacity_kwh = capacity_kwh

        # Current stored battery energy.
        self.soc_kwh = initial_soc_kwh
       
        # Physical minimum stored energy permitted by the battery
        # model. This is separate from any controller-level SOC
        # reserve maintained for VPP operation.
        self.min_soc_kwh = min_soc_kwh


        # ----------------------------------------------------
        # BATTERY POWER LIMITS
        # ----------------------------------------------------

        # Maximum continuous charge and discharge powers after
        # accounting for the installed battery/inverter limits
        # specified when each Battery object is created.
        self.max_charge_kw = max_charge_kw
        self.max_discharge_kw = max_discharge_kw


        # ----------------------------------------------------
        # BATTERY EFFICIENCY
        # ----------------------------------------------------

        # Efficiency applied when converting between electrical
        # energy at the battery interface and stored battery energy.
        self.efficiency = efficiency


        # ----------------------------------------------------
        # BATTERY METADATA
        # ----------------------------------------------------

        # Equipment information retained for result logging,
        # reporting and demonstrator documentation.
        self.manufacturer = manufacturer
        self.model = model
        self.module_count = module_count
        
    def charge(
        self,
        requested_power_kw,
        timestep_hours
    ):
        """
        Attempt to charge the battery at the requested electrical
        power for one simulation timestep.

        Charging is limited by:
            1. the battery maximum charge power,
            2. the remaining battery energy capacity.

        Returns signed battery power:
            negative = charging
            zero     = no charging
        """

        # Ignore zero or negative charge requests.
        if requested_power_kw <= 0:
            return 0.0


        # ----------------------------------------------------
        # APPLY CHARGE POWER LIMIT
        # ----------------------------------------------------

        # Restrict the requested electrical charging power to the
        # battery's maximum permitted charge power.
        charge_power_kw = min(
            requested_power_kw,
            self.max_charge_kw
        )


        # ----------------------------------------------------
        # CONVERT INPUT POWER TO STORED ENERGY
        # ----------------------------------------------------

        # Determine the energy that would be added to the battery
        # during this timestep after applying charging efficiency.
        requested_energy_kwh = (
            charge_power_kw
            * timestep_hours
            * self.efficiency
        )


        # ----------------------------------------------------
        # APPLY BATTERY ENERGY-CAPACITY LIMIT
        # ----------------------------------------------------

        # Remaining energy that can be stored before the battery
        # reaches its usable capacity.
        available_space_kwh = (
            self.capacity_kwh
            - self.soc_kwh
        )

        # Actual stored energy cannot exceed the remaining
        # available battery capacity.
        actual_energy_kwh = min(
            requested_energy_kwh,
            available_space_kwh
        )


        # ----------------------------------------------------
        # UPDATE BATTERY SOC
        # ----------------------------------------------------

        self.soc_kwh += actual_energy_kwh


        # ----------------------------------------------------
        # DETERMINE ACTUAL ELECTRICAL CHARGING POWER
        # ----------------------------------------------------

        # Convert the energy actually stored back to the
        # corresponding electrical charging power at the battery
        # interface.
        actual_charge_power_kw = (
            actual_energy_kwh
            / (
                timestep_hours
                * self.efficiency
            )
        )

        # House uses negative battery power to represent charging.
        battery_power_kw = (
            -actual_charge_power_kw
        )

        return battery_power_kw
        
    def discharge(
        self,
        requested_power_kw,
        timestep_hours
    ):
        """
        Attempt to discharge the battery at the requested
        electrical power for one simulation timestep.

        Discharging is limited by:
            1. the battery maximum discharge power,
            2. the energy available above physical minimum SOC.

        Returns signed battery power:
            positive = discharging
            zero     = no discharge
        """

        # Ignore zero or negative discharge requests.
        if requested_power_kw <= 0:
            return 0.0


        # ----------------------------------------------------
        # APPLY DISCHARGE POWER LIMIT
        # ----------------------------------------------------

        # Restrict requested output power to the battery's
        # maximum permitted discharge power.
        discharge_power_kw = min(
            requested_power_kw,
            self.max_discharge_kw
        )


        # ----------------------------------------------------
        # CONVERT OUTPUT POWER TO REQUIRED BATTERY ENERGY
        # ----------------------------------------------------

        # Determine how much stored battery energy must be removed
        # to provide the requested electrical output after losses.
        requested_energy_kwh = (
            discharge_power_kw
            * timestep_hours
            / self.efficiency
        )


        # ----------------------------------------------------
        # APPLY MINIMUM SOC / AVAILABLE ENERGY LIMIT
        # ----------------------------------------------------

        # Stored energy physically available for discharge above
        # the battery model's minimum SOC.
        available_energy_kwh = max(
            0.0,
            self.soc_kwh
            - self.min_soc_kwh
        )

        # Actual battery energy removed cannot exceed the energy
        # currently available above minimum SOC.
        actual_energy_kwh = min(
            available_energy_kwh,
            requested_energy_kwh
        )


        # ----------------------------------------------------
        # UPDATE BATTERY SOC
        # ----------------------------------------------------

        self.soc_kwh -= actual_energy_kwh


        # ----------------------------------------------------
        # DETERMINE ACTUAL ELECTRICAL DISCHARGE POWER
        # ----------------------------------------------------

        # Apply discharge efficiency to determine the electrical
        # power actually delivered by the battery.
        actual_discharge_power_kw = (
            actual_energy_kwh
            * self.efficiency
            / timestep_hours
        )

        # Positive battery power represents discharge.
        battery_power_kw = (
            actual_discharge_power_kw
        )

        return battery_power_kw
        
    def get_soc_percent(self):
        """
        Return the current battery state of charge as a percentage
        of the modelled usable battery capacity.
        """

        return (
            self.soc_kwh
            / self.capacity_kwh
        ) * 100


    def get_soc_kwh(self):
        """
        Return the current stored battery energy in kWh.
        """

        return self.soc_kwh

# ============================================================
# VPP SUPERVISORY CONTROLLER
# ============================================================

class VPPController:
    """
    Supervisory controller for the five-house residential VPP.

    The controller does not directly modify battery SOC or final
    premises power flows. Instead, it:

        1. evaluates the current aggregate VPP state,
        2. selects the active VPP operating mode,
        3. allocates network-support battery targets,
        4. coordinates available surplus PV between houses, and
        5. returns supervisory signals to each House object.

    House and Battery objects remain responsible for enforcing
    physical DER and connection-point constraints.

    Current operating priority:
        1. explicit network-support request,
        2. manual/test override,
        3. scheduled cheap-energy charging,
        4. scheduled peak shaving,
        5. grid-import-limit response,
        6. normal/default operation.

    PV sharing is treated as a secondary VPP action rather than
    a standalone operating mode.
    """

    def __init__(
        self,
        default_mode="self_consumption",
        peak_hours=None,
        cheap_energy_hours=None,
        grid_import_limit_kw=None
    ):

        # Normal operating mode when no higher-priority
        # supervisory action is required.
        self.default_mode = default_mode

        # Time periods during which scheduled peak-shaving
        # behaviour may be enabled.
        self.peak_hours = peak_hours or []

        # Time periods during which the current placeholder
        # arbitrage strategy may charge batteries from the grid.
        self.cheap_energy_hours = (
            cheap_energy_hours or []
        )

        # Optional aggregate VPP import threshold.
        # If exceeded, the controller can activate peak shaving.
        self.grid_import_limit_kw = (
            grid_import_limit_kw
        )
        
    def get_signals(
        self,
        t,
        external_inputs=None,
        house_states=None
    ):
        """
        Evaluate the current VPP state and generate supervisory
        control signals for one simulation timestep.

        Parameters
        ----------
        t:
            Current simulation hour-of-day.

        external_inputs:
            External supervisory commands such as network-support,
            PV-sharing, peak-shaving or manual-mode requests.

        house_states:
            Pre-control state and available flexibility reported
            by each participating House object.
        """

        if external_inputs is None:
            external_inputs = {}

        if house_states is None:
            house_states = []


        # ----------------------------------------------------
        # AGGREGATE PRE-CONTROL VPP STATE
        # ----------------------------------------------------
        #
        # The VPP consists only of the five participating houses
        # and their DERs. Hospital and wind/DFIG resources are
        # external to this controller boundary.

        # Aggregate premises power before battery dispatch.
        #
        # Positive = net VPP grid import.
        # Negative = net VPP grid export.
        total_house_net_before_battery_kw = sum(
            state["net_power_kw_before_battery"]
            for state in house_states
        )

        vpp_net_power_before_battery_kw = (
            total_house_net_before_battery_kw
        )

        vpp_grid_import_before_battery_kw = max(
            vpp_net_power_before_battery_kw,
            0.0
        )


        # ----------------------------------------------------
        # AGGREGATE PV-SHARING FLEXIBILITY
        # ----------------------------------------------------

        # Genuine surplus PV remaining after source houses have
        # satisfied their own demand and local battery capability.
        total_shared_pv_available_kw = sum(
            state["shared_pv_available_kw"]
            for state in house_states
        )

        # Total ability of participating houses to receive shared
        # PV through either unmet demand or available battery
        # charging capability.
        total_shared_pv_receiving_capacity_kw = sum(
            state["shared_pv_receiving_capacity_kw"]
            for state in house_states
        )

        

        # ----------------------------------------------------
        # EXTERNAL SUPERVISORY INPUTS
        # ----------------------------------------------------

        # Explicit request for the VPP to provide network support.
        network_support_request = external_inputs.get(
            "network_support_request",
            False
        )

        # Aggregate battery support power requested from the VPP.
        requested_support_kw = external_inputs.get(
            "requested_support_kw",
            0.0
        )

        # Enables scheduled peak-shaving behaviour.
        peak_shaving_enabled = external_inputs.get(
            "peak_shaving_enabled",
            False
        )

        # Enables secondary VPP surplus-PV sharing.
        pv_sharing_enabled = external_inputs.get(
            "pv_sharing_enabled",
            False
        )

        # Direct operating-mode override retained for testing
        # and physical demonstrator development.
        manual_mode = external_inputs.get(
            "manual_mode"
        )

        # Enables deliberate battery export during the
        # configured high-value peak period.
        economic_export_enabled = external_inputs.get(
            "economic_export_enabled",
            False
        )

        # ----------------------------------------------------
        # INITIALISE HOUSE-LEVEL CONTROL TARGETS
        # ----------------------------------------------------

        # Battery discharge targets used during explicit
        # network-support operation.
        house_dispatch_targets = {}

        # Shared PV allocated to receiving VPP houses.
        house_shared_pv_targets = {}

        # A09:
        # Shared-PV export attributed back to the VPP houses
        # supplying the transferred surplus energy.
        house_shared_export_targets = {} 
        
        # Aggregate battery discharge capability currently
        # available to the VPP after physical battery and
        # connection constraints reported by each house.
        total_available_discharge_kw = sum(
            state["vpp_available_discharge_kw"]
            for state in house_states
        )
        
        # ====================================================
        # PRIMARY VPP OPERATING-MODE SELECTION
        # ====================================================
        #
        # Higher-priority conditions are evaluated first.
        # Only one primary operating mode is selected for the
        # current timestep.


        # ----------------------------------------------------
        # PRIORITY 1 - EXPLICIT NETWORK SUPPORT
        # ----------------------------------------------------

        if network_support_request:

            active_mode = "network_support"

            # Select battery-equipped houses that currently have
            # usable VPP discharge capability.
            available_houses = [
                state
                for state in house_states
                if state["has_battery"]
                and state["vpp_available_discharge_kw"] > 0
            ]

            # Remaining aggregate support request that has not yet
            # been allocated among participating batteries.
            remaining_support_kw = (
                requested_support_kw
            )

            # Iteratively divide the remaining request equally
            # between houses that still have discharge capacity.
            #
            # If one house reaches its available limit, its unused
            # share is redistributed among the remaining houses.
            while (
                available_houses
                and remaining_support_kw > 0
            ):

                equal_share_kw = (
                    remaining_support_kw
                    / len(available_houses)
                )

                still_available_houses = []

                for state in available_houses:

                    house_name = state["name"]

                    available_kw = state[
                        "vpp_available_discharge_kw"
                    ]

                    # Support already allocated to this house
                    # during a previous redistribution pass.
                    already_dispatched_kw = (
                        house_dispatch_targets.get(
                            house_name,
                            0.0
                        )
                    )

                    remaining_house_capacity_kw = (
                        available_kw
                        - already_dispatched_kw
                    )

                    dispatch_kw = min(
                        equal_share_kw,
                        remaining_house_capacity_kw
                    )

                    house_dispatch_targets[house_name] = (
                        already_dispatched_kw
                        + dispatch_kw
                    )

                    remaining_support_kw -= dispatch_kw
                    remaining_house_capacity_kw -= dispatch_kw

                    # Retain houses that can accept another share
                    # during the next redistribution pass.
                    if remaining_house_capacity_kw > 0:
                        still_available_houses.append(
                            state
                        )

                available_houses = (
                    still_available_houses
                )
                
       
        # ----------------------------------------------------
        # PRIORITY 2 - MANUAL / TEST OVERRIDE
        # ----------------------------------------------------

        elif manual_mode is not None:
            active_mode = manual_mode


        # ----------------------------------------------------
        # PRIORITY 3 - CHEAP-ENERGY CHARGING
        # ----------------------------------------------------
        #
        # Current placeholder for later tariff/economic
        # optimisation.

        elif t in self.cheap_energy_hours:
            active_mode = "arbitrage_charge"
        
        # ----------------------------------------------------
        # PRIORITY 4 - ECONOMIC PEAK EXPORT
        # ----------------------------------------------------
        
        elif (
            economic_export_enabled
            and t in self.peak_hours
        ):
            active_mode = "economic_export"

        
        # ----------------------------------------------------
        # PRIORITY 5 - SCHEDULED PEAK SHAVING
        # ----------------------------------------------------

        elif (
            peak_shaving_enabled
            and t in self.peak_hours
        ):
            active_mode = "peak_shaving"



        # ----------------------------------------------------
        # PRIORITY 6 - VPP IMPORT-LIMIT RESPONSE
        # ----------------------------------------------------

        elif (
            self.grid_import_limit_kw is not None
            and vpp_grid_import_before_battery_kw
            > self.grid_import_limit_kw
        ):
            active_mode = "peak_shaving"


        # ----------------------------------------------------
        # PRIORITY 7 - NORMAL VPP OPERATION
        # ----------------------------------------------------

        else:
            active_mode = self.default_mode

            
        # ====================================================
        # SECONDARY VPP ACTION - SURPLUS PV SHARING
        # ====================================================
        #
        # PV sharing is independent of the primary operating-mode
        # selection and is therefore treated as a secondary action.
        #
        # Sharing is currently permitted during normal
        # self-consumption and non-active peak-shaving operation,
        # but is disabled during explicit peak periods.

        shared_pv_transfer_allowed = (
            pv_sharing_enabled
            and active_mode in [
                "self_consumption",
                "peak_shaving"
            ]
            and not (
                peak_shaving_enabled
                and t in self.peak_hours
            )
        )
        
        if (
            shared_pv_transfer_allowed
            and total_shared_pv_available_kw > 0
            and total_shared_pv_receiving_capacity_kw > 0
        ):

            # Select every participating house that can currently
            # receive shared PV through unmet household demand,
            # battery charging capacity, or both.
            available_houses = [
                state
                for state in house_states
                if state[
                    "shared_pv_receiving_capacity_kw"
                ] > 0
            ]

            remaining_shared_pv_kw = (
                total_shared_pv_available_kw
            )

            # Allocate available shared PV equally between
            # receiving houses, redistributing unused shares when
            # an individual house reaches its receiving capacity.
            while (
                available_houses
                and remaining_shared_pv_kw > 0
            ):

                equal_share_kw = (
                    remaining_shared_pv_kw
                    / len(available_houses)
                )

                still_available_houses = []

                for state in available_houses:

                    house_name = state["name"]

                    available_kw = state[
                        "shared_pv_receiving_capacity_kw"
                    ]

                    already_allocated_kw = (
                        house_shared_pv_targets.get(
                            house_name,
                            0.0
                        )
                    )

                    remaining_house_capacity_kw = (
                        available_kw
                        - already_allocated_kw
                    )

                    # Shared-PV allocation may serve either
                    # household demand or battery charging.
                    allocation_kw = min(
                        equal_share_kw,
                        remaining_house_capacity_kw
                    )

                    house_shared_pv_targets[
                        house_name
                    ] = (
                        already_allocated_kw
                        + allocation_kw
                    )

                    remaining_shared_pv_kw -= (
                        allocation_kw
                    )

                    remaining_house_capacity_kw -= (
                        allocation_kw
                    )

                    if remaining_house_capacity_kw > 0:
                        still_available_houses.append(
                            state
                        )

                available_houses = (
                    still_available_houses
                ) 

        # ====================================================
        # A09 - ACTUAL SHARED-PV TRANSFER
        # ====================================================

        # Total PV actually allocated to receiving VPP houses.
        # This may be lower than the total PV available for sharing
        # if receiving demand/storage capacity is insufficient.
        total_shared_pv_transferred_kw = sum(
            house_shared_pv_targets.values()
        )


        # ----------------------------------------------------
        # ATTRIBUTE SHARED PV TO SOURCE HOUSES
        # ----------------------------------------------------
        #
        # Attribute the transferred energy back to houses with
        # genuine available PV surplus.
        #
        # These source targets are subsequently used by House.step()
        # when applying the A09 shared-PV-specific export allowance.

        remaining_shared_export_kw = (
            total_shared_pv_transferred_kw
        )

        for state in house_states:

            if remaining_shared_export_kw <= 0:
                break

            source_available_kw = state[
                "shared_pv_available_kw"
            ]

            if source_available_kw <= 0:
                continue

            source_export_kw = min(
                source_available_kw,
                remaining_shared_export_kw
            )

            house_shared_export_targets[
                state["name"]
            ] = source_export_kw

            remaining_shared_export_kw -= (
                source_export_kw
            )


        # ====================================================
        # RETURN SUPERVISORY CONTROL SIGNALS
        # ====================================================

        return {
            # Primary operating state
            "mode": active_mode,

            "is_peak_period":
                t in self.peak_hours,

            "is_grid_peak_limit_exceeded": (
                self.grid_import_limit_kw is not None
                and vpp_grid_import_before_battery_kw
                > self.grid_import_limit_kw
            ),

            "peak_shaving_enabled":
                peak_shaving_enabled,

            "pv_sharing_enabled":
                pv_sharing_enabled,
            "economic_export_enabled":
                economic_export_enabled,

            # Network-support event state.
            "is_event_period":
                network_support_request,

            "is_cheap_energy_period":
                t in self.cheap_energy_hours,

            # Network-support request and house allocations
            "requested_support_kw":
                requested_support_kw,

            "house_dispatch_targets":
                house_dispatch_targets,

            # Aggregate pre-control VPP state
            "vpp_grid_import_before_battery_kw":
                vpp_grid_import_before_battery_kw,

            # Current VPP battery capability
            "total_available_discharge_kw":
                total_available_discharge_kw,

            # Shared-PV state
            "total_shared_pv_transferred_kw":
                total_shared_pv_transferred_kw,

            "total_shared_pv_available_kw":
                total_shared_pv_available_kw,

            "total_shared_pv_receiving_capacity_kw":
                total_shared_pv_receiving_capacity_kw,

            # House-level shared-PV allocations
            "house_shared_pv_targets":
                house_shared_pv_targets,

            "house_shared_export_targets":
                house_shared_export_targets,
        }
        


# ============================================================
# EXTERNAL LOCAL LOAD MODEL
# ============================================================

class LocalLoad:
    """
    Represents an electrical load located outside the VPP
    participant boundary.

    In the current demonstrator this object represents the
    hospital load.

    The load contributes to the wider demonstration/network
    power balance but is not a participating VPP asset and is
    therefore not directly controlled by the VPP controller.

    The model uses a time-series demand profile so the external
    load can later be expanded without changing the main
    simulation architecture.
    """

    def __init__(
        self,
        name,
        demand_kw
    ):

        # Descriptive name used in simulation results.
        self.name = name

        # Time-series external demand profile in kW.
        self.demand_kw = demand_kw


    def step(self, t):
        """
        Return the external load demand for the current
        simulation timestep.
        """

        return {
            "name": self.name,
            "load_kw": self.demand_kw[t]
        }
# ============================================================
# EXTERNAL WIND GENERATION MODEL
# ============================================================

class WindTurbine:
    """
    Represents the external wind/DFIG generation element used
    by the physical demonstrator.

    Wind generation is outside the five-house VPP boundary.
    It contributes only to the wider demonstration/network
    power balance.

    The current model uses a predefined generation profile.
    More detailed wind-speed, turbine or DFIG behaviour can be
    added later if required by the demonstrator scope.
    """

    def __init__(
        self,
        name,
        generation_profile
    ):

        # Descriptive name used in simulation results.
        self.name = name

        # Time-series wind generation profile in kW.
        self.generation_profile = (
            generation_profile
        )


    def step(self, t):
        """
        Return external wind generation for the current
        simulation timestep.
        """

        return {
            "name": self.name,
            "generation_kw":
                self.generation_profile[t]
        }





In [4]:
def get_hardware_states(simulation):
    """
    Convert the latest simulated battery results
    into states for the physical LEDs.
    """

    states = {}

    for house in simulation.houses:

        # House 3 has no battery
        if house.battery is None:
            states[house.name] = {
                "state": "no_battery",
                "soc_percent": 0.0
            }
            
            continue

        # Get this house's result from the timestep
        # that has just been simulated
        latest = simulation.house_results_log[house.name][-1]

        soc_kwh = latest["battery_soc_kwh"]
        charge_kw = latest["battery_charge_kw"]
        discharge_kw = latest["battery_discharge_kw"]
        soc_percent = latest["battery_soc_percent"]

        max_soc_kwh = house.battery.capacity_kwh
        min_soc_kwh = house.battery.min_soc_kwh

        # Small tolerance avoids floating-point comparison problems
        tolerance = 1e-6

        if soc_kwh >= max_soc_kwh - tolerance:
            state = "full"

        elif soc_kwh <= min_soc_kwh + tolerance:
            state = "minimum"

        elif charge_kw > tolerance:
            state = "charging"

        elif discharge_kw > tolerance:
            state = "discharging"

        else:
            state = "idle"

        states[house.name] = {
            "state": state,
            "soc_percent": soc_percent
        }

    return states

Simulation Loop

In [5]:
# ============================================================
# REALISTIC PROFILE DATA
# ============================================================

# Simulation resolution
timestep_hours = 0.5
num_steps = len(summer_data)
simulation_hours = num_steps * timestep_hours



# ============================================================
# EXTERNAL DEMONSTRATION PLACEHOLDER PROFILES
# ============================================================

# Hospital placeholder - constant demand for now
hospital_load_profile = np.full(
    num_steps,
    6.0
)

# Wind placeholder - zero generation for now
wind_generation_profile = np.zeros(
    num_steps
)



# ----------------------------
# Household load profiles
# ----------------------------

house_1_load = summer_data["H1_HI_B_kW"].to_numpy()
house_2_load = summer_data["H2_MED_B_kW"].to_numpy()
house_3_load = summer_data["H3_LOW_A_Kw"].to_numpy()
house_4_load = summer_data["H4_Hi_A_kW"].to_numpy()
house_5_load = summer_data["H5_Med_A_kW"].to_numpy()


# ----------------------------
# PV generation profiles
# ----------------------------

house_1_pv = summer_data["PV_13.3kW_kW"].to_numpy()
house_2_pv = summer_data["PV_6.6kW_kW"].to_numpy()
house_3_pv = summer_data["PV_13.3kW_kW"].to_numpy()
house_4_pv = summer_data["PV_20kW_kW"].to_numpy()

# House 5 has no PV
house_5_pv = np.zeros(num_steps)


load_timestamps = pd.to_datetime(
    summer_data["LOAD_SOURCE_TIMESTEP"],
    dayfirst=True
)


pv_timestamps = pd.to_datetime(
    summer_data["PV_SOURCE_TIMESTAMP"],
    dayfirst=True
)

print("\n=== TIMESTAMP CHECK ===")
print("Load start:", load_timestamps.iloc[0])
print("Load end:  ", load_timestamps.iloc[-1])
print("PV start:  ", pv_timestamps.iloc[0])
print("PV end:    ", pv_timestamps.iloc[-1])

#need to add Wind Turbine and industrial load profiles here potentially? UPDATE WIND GEN AND LOCAL LOAD CLASS WHEN THIS IS DONE

# Verify all profiles are the same length
profiles = [
    house_1_load,
    house_2_load,
    house_3_load,
    house_4_load,
    house_5_load,
    house_1_pv,
    house_2_pv,
    house_3_pv,
    house_4_pv,
    house_5_pv,
    hospital_load_profile,
    wind_generation_profile,
]
profile_lengths = set([len(profile) for profile in profiles])


if len(profile_lengths) > 1 :
    raise ValueError("Profiles are not the same length")
else :
    print("All Load and Generation Profiles are the same length")

print("Number of Simulation steps=", num_steps)
print("Number of Hours Simulated=", simulation_hours) 

# --------------------------------------------------
# SigenStor battery simulation assumptions
# --------------------------------------------------
#
# Battery model based on Sigen Battery 9.0 module:
#   Usable energy per module: 8.76 kWh
#   Continuous charge/discharge power per module: 4.6 kW
#
# Simulation assumptions:
#   - Module energy and power capability are cumulative.
#   - BESS inverter charge/discharge power is capped at 10 kW.
#   - Manufacturer usable capacity represents the simulated
#     0–100% SOC operating range.
#   - Any future VPP SOC reserve will be implemented by the
#     controller, not by reducing physical battery capacity.

SIGEN_MODULE_USABLE_KWH = 8.76
SIGEN_MODULE_POWER_KW = 4.6

BESS_INVERTER_LIMIT_KW = 10.0


def create_sigenstor_spec(module_count, initial_soc_percent=50.0):

    usable_capacity_kwh = (
        module_count * SIGEN_MODULE_USABLE_KWH
    )

    raw_stack_power_kw = (
        module_count * SIGEN_MODULE_POWER_KW
    )

    max_bess_power_kw = min(
        raw_stack_power_kw,
        BESS_INVERTER_LIMIT_KW
    )

    initial_soc_kwh = (
        usable_capacity_kwh
        * initial_soc_percent / 100.0
    )

    return {
        "capacity_kwh": usable_capacity_kwh,
        "initial_soc_kwh": initial_soc_kwh,

        "max_charge_kw": max_bess_power_kw,
        "max_discharge_kw": max_bess_power_kw,

        "efficiency": 0.95,

        # 0% represents the bottom of the manufacturer's
        # stated usable energy range.
        "min_soc_kwh": 0.0,
        
        "manufacturer": "Sigenergy",
        "model": "SigenStor BAT 9.0",
        "module_count": module_count,
    }



=== TIMESTAMP CHECK ===
Load start: 2013-02-04 00:00:00
Load end:   2013-02-10 23:30:00
PV start:   2013-01-07 00:00:00
PV end:     2013-01-13 23:30:00
All Load and Generation Profiles are the same length
Number of Simulation steps= 336
Number of Hours Simulated= 168.0


In [6]:

# Self-consumption is the normal operating mode.
# Tariff peak shaving can be enabled through external_inputs.

controller = VPPController(
    default_mode="self_consumption",
    peak_hours=[
        16.0, 16.5,
        17.0, 17.5,
        18.0, 18.5,
        19.0, 19.5,
        20.0, 20.5,
    ],
    cheap_energy_hours=[]
)
scenario_name="external_network_support_test"


# Initialise Batteries

# --------------------------------------------------
# Initialise SigenStor battery systems
# --------------------------------------------------

# H1 - High-DER prosumer
# 3 x Sigen Battery 9.0 modules
house1_battery = Battery(
    **create_sigenstor_spec(
        module_count=3,
        initial_soc_percent=50.0
    )
)

# H2 - Typical prosumer
# 2 x Sigen Battery 9.0 modules
house2_battery = Battery(
    **create_sigenstor_spec(
        module_count=2,
        initial_soc_percent=50.0
    )
)

# H3 - PV-only household
# No battery

# H4 - High-demand / future three-phase household
# 3 x Sigen Battery 9.0 modules
house4_battery = Battery(
    **create_sigenstor_spec(
        module_count=3,
        initial_soc_percent=50.0
    )
)

# H5 - Battery-heavy / no-PV household
# 5 x Sigen Battery 9.0 modules
house5_battery = Battery(
    **create_sigenstor_spec(
        module_count=5,
        initial_soc_percent=50.0
    )
)



# Initialise Houses
houses = [
   
    # H1 - High-DER prosumer
    House(
        name="House 1",
        load_profile=house_1_load,
        generation_profile=house_1_pv,
        battery=house1_battery,
        pv_array_kwp=13.3,
        pv_inverter_limit_kw=10.64,
        connection_type="single_phase",
        connection_phases=1
    ),

    # H2 - Typical prosumer
    House(
        name="House 2",
        load_profile=house_2_load,
        generation_profile=house_2_pv,
        battery=house2_battery,
        pv_array_kwp=6.6,
        pv_inverter_limit_kw=5.28,
        connection_type="single_phase",
        connection_phases=1
    ),

    # H3 - PV-rich household, no battery
    House(
        name="House 3",
        load_profile=house_3_load,
        generation_profile=house_3_pv,
        battery=None,
        pv_array_kwp=13.3,
        pv_inverter_limit_kw=10.64,
        connection_type="single_phase",
        connection_phases=1
    ),

    # H4 - High-demand three-phase household
    House(
        name="House 4",
        load_profile=house_4_load,
        generation_profile=house_4_pv,
        battery=house4_battery,
        pv_array_kwp=20.0,
        pv_inverter_limit_kw=16.0,
        connection_type="three_phase",
        connection_phases=3
    ),

    # H5 - Battery-only household
    House(
        name="House 5",
        load_profile=house_5_load,
        generation_profile=house_5_pv,
        battery=house5_battery,
        pv_array_kwp=0.0,
        pv_inverter_limit_kw=0.0,
        connection_type="single_phase",
        connection_phases=1
    ),
]
# TEMPORARY CONNECTION-POINT TEST
# houses[0].current_export_limit_kw = 1.0

# houses[2].dynamic_export_limit_kw = 1.0
# houses[2].current_export_limit_kw = 1.0

# Initialise hospital and wind turbine

hospital = LocalLoad(
    "Hospital",
    hospital_load_profile
)

wind = WindTurbine(
    "DFIG Wind Placeholder",
    wind_generation_profile
)

# Create simulation
simulation = VPPSimulation(
    houses=houses,
    hospital=hospital,
    wind=wind,
    controller=controller,
    timestep_hours=timestep_hours,
    num_steps=num_steps
)

In [8]:
# --------------------------------------------------
# Continuous hardware demonstration
# --------------------------------------------------


# --------------------------------------------------
# Simulated network support event
# --------------------------------------------------

# event_day = 1
# event_hour = 22
# event_duration_hours = 2

# event_start_step = int(
#     (((event_day - 1) * 24) + event_hour)
#     / timestep_hours
# )

# event_duration_steps = int(
#     event_duration_hours / timestep_hours
# )

# event_end_step = (
#     event_start_step
#     + event_duration_steps
# )

# --------------------------------------------------
# Simulated tariff peak-shaving toggle
# --------------------------------------------------

peak_toggle_day = 2
peak_toggle_hour = 10
peak_toggle_duration_hours = 12

peak_toggle_start_step = int(
    (((peak_toggle_day - 1) * 24) + peak_toggle_hour)
    / timestep_hours
)

peak_toggle_duration_steps = int(
    peak_toggle_duration_hours / timestep_hours
)

peak_toggle_end_step = (
    peak_toggle_start_step
    + peak_toggle_duration_steps
)

hardware = HardwareInterface()

try:
    print("Continuous VPP hardware demonstration started.")
    print("Use Jupyter interrupt/stop to finish the run.\n")

    while True:
        
        # Read physical ESP32 inputs first
        external_inputs = hardware.read_inputs()
        
        # ----------------------------------------------
        # ECONOMIC PEAK EXPORT
        # ----------------------------------------------
        #
        # Temporary demonstration input.
        # Enables deliberate battery export during the
        # peak hours configured in VPPController.
        
        external_inputs["economic_export_enabled"] = True

        
        # ----------------------------------------------
        # DAILY REPEATING PV SHARING
        # ----------------------------------------------
    
        current_hour_of_day = (
            simulation.current_step * timestep_hours
        ) % 24
    
        pv_sharing_enabled = (
            10.0 <= current_hour_of_day < 15.0
        )
    
        external_inputs["pv_sharing_enabled"] = (
            pv_sharing_enabled
        )
      
        # ----------------------------------------------
        # SIMULATED PEAK-SHAVING STRATEGY
        # ----------------------------------------------
        
        # Simulated peak-shaving strategy button
        peak_shaving_enabled = (
            peak_toggle_start_step
            <= simulation.current_step
            < peak_toggle_end_step
        )
        
        external_inputs["peak_shaving_enabled"] = peak_shaving_enabled
       

        # # Simulated network-support event
        # if (
        #     event_start_step
        #     <= simulation.current_step
        #     < event_end_step
        # ):
        #     external_inputs["network_support_request"] = True
        #     external_inputs["requested_support_kw"] = 28.0
        # Run one logical simulation timestep
        result = simulation.step(external_inputs)

      
        # Convert latest simulation results
        # into LED states
        hardware_states = get_hardware_states(simulation)

        

        hardware.update_outputs(
            result,
            hardware_states
        )
        

        # Print useful live information
        print(
            f"\nStep {result['Simulation_step']} | "
            f"Day {result['Day_number']} | "
            f"Hour {result['Hour_of_day']:.1f} | "
            f"Mode: {result['VPP_mode']} | "
            f"{'ON' if external_inputs.get('network_support_request', False) else 'OFF'} | "
            f"Peak strategy: {'ON' if peak_shaving_enabled else 'OFF'} | "
            f"PV sharing: {'ON' if result['PV_sharing_enabled'] else 'OFF'} | "
        )

        for house in simulation.houses:

            latest = simulation.house_results_log[house.name][-1]
            state = hardware_states[house.name]["state"]

            print(
                f"{house.name}: "
                f"{state:12s} | "
                f"SOC = {latest['battery_soc_kwh']:.2f} kWh "
                f"({latest['battery_soc_percent']:.1f}% |"
                f"Charge = {latest['battery_charge_kw']:.2f} kW | "
                f"Discharge = {latest['battery_discharge_kw']:.2f} kW"
            )
        print(
            "Shared PV available: "
            f"{result['Shared_PV_available_kw']:.2f} kW | "
            f"Shared PV targets: {result['house_shared_pv_targets']}"
        )

        # Extra output during network-support event
        if result["VPP_mode"] == "network_support":

            print(
                "NETWORK SUPPORT | "
                f"Requested = {result['Requested_support_kw']:.2f} kW | "
                f"Available = {result['Available_VPP_discharge_kw']:.2f} kW | "
                f"Actual = {result['Battery_discharge_kw']:.2f} kW"
            )


            
        time.sleep(1)


except KeyboardInterrupt:

    print("\nSimulation interrupted by user.")


finally:

    hardware.cleanup()

    print("GPIO outputs cleaned up.")


# --------------------------------------------------
# Export everything accumulated during the run
# --------------------------------------------------

timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M")

filename = f"SimulationResults_{scenario_name}_{timestamp}.xlsx"
filepath = f"results/{filename}"

summary_df = pd.DataFrame(
    simulation.simulation_results_summary
).round(2)

with pd.ExcelWriter(filepath, engine="openpyxl") as writer:

    summary_df.to_excel(
        writer,
        sheet_name="Summary",
        index=False
    )

    for house_name, house_log in simulation.house_results_log.items():

        house_df = pd.DataFrame(house_log).round(2)

        sheet_name = house_name.replace(" ", "_")

        house_result_export = house_df.drop(
            columns=["name"]
        )

        house_result_export.to_excel(
            writer,
            sheet_name=sheet_name,
            index=False
        )

print(
    simulation.current_step,
    "simulation steps completed"
)

print(
    f"Results exported to results/{filename}"
)

The history saving thread hit an unexpected error (OperationalError('disk I/O error')).History will not be written to the database.
Continuous VPP hardware demonstration started.
Use Jupyter interrupt/stop to finish the run.


Step 0 | Day 1 | Hour 0.0 | Mode: self_consumption | OFF | Peak strategy: OFF | PV sharing: OFF | 
House 1: discharging  | SOC = 12.77 kWh (48.6% |Charge = -0.00 kW | Discharge = 0.71 kW
House 2: discharging  | SOC = 8.68 kWh (49.5% |Charge = -0.00 kW | Discharge = 0.15 kW
House 3: no_battery   | SOC = 0.00 kWh (0.0% |Charge = -0.00 kW | Discharge = 0.00 kW
House 4: discharging  | SOC = 13.03 kWh (49.6% |Charge = -0.00 kW | Discharge = 0.21 kW
House 5: discharging  | SOC = 21.71 kWh (49.6% |Charge = -0.00 kW | Discharge = 0.35 kW
Shared PV available: 0.00 kW | Shared PV targets: {}

Step 1 | Day 1 | Hour 0.5 | Mode: self_consumption | OFF | Peak strategy: OFF | PV sharing: OFF | 
House 1: discharging  | SOC = 12.40 kWh (47.2% |Charge = -0.00 kW | Discharge = 0.69

In [54]:

# --------------------------------------------------
# REAL PROFILE DEBUG
# --------------------------------------------------

debug_step = 27



for house in simulation.houses:

    result = simulation.house_results_log[
        house.name
    ][debug_step]

    print(
        f"\n{house.name}"
        f"\n  Load:                 {result['load_kw']:.3f} kW"
        f"\n  Available PV:         {result['available_generation_kw']:.3f} kW"
        f"\n  Net before battery:   {result['net_power_kw_before_battery']:.3f} kW"
        f"\n  Battery charge:       {result['battery_charge_kw']:.3f} kW"
        f"\n  Battery discharge:    {result['battery_discharge_kw']:.3f} kW"
        f"\n  Final net power:      {result['net_power_kw']:.3f} kW"
       
    )

print(
    "\nShared PV targets:",
    simulation.simulation_results_summary[
        debug_step
    ]["house_shared_pv_targets"]
)

debug_summary = (
    simulation.simulation_results_summary[
        debug_step
    ]
)

print("\nVPP CHARGE AVAILABILITY:")

for house_name, available_charge in (
    debug_summary[
        "VPP_available_charge_kw"
    ].items()
):
    print(
        f"{house_name}: "
        f"{available_charge:.3f} kW"
    )

print(
    "\nShared PV targets:",
    debug_summary["house_shared_pv_targets"]
)



=== STEP 22 / 11:00 HOUSE DEBUG ===

House 1
  Load:                 0.434 kW
  Available PV:         8.100 kW
  Net before battery:   -7.666 kW
  Battery charge:       0.000 kW
  Battery discharge:    -0.000 kW
  Final net power:      -7.666 kW

House 2
  Load:                 0.213 kW
  Available PV:         4.020 kW
  Net before battery:   -3.807 kW
  Battery charge:       0.000 kW
  Battery discharge:    -0.000 kW
  Final net power:      -3.807 kW

House 3
  Load:                 0.253 kW
  Available PV:         8.100 kW
  Net before battery:   -7.847 kW
  Battery charge:       -0.000 kW
  Battery discharge:    0.000 kW
  Final net power:      -7.847 kW

House 4
  Load:                 0.948 kW
  Available PV:         12.181 kW
  Net before battery:   -11.233 kW
  Battery charge:       0.000 kW
  Battery discharge:    -0.000 kW
  Final net power:      -11.233 kW

House 5
  Load:                 0.332 kW
  Available PV:         0.000 kW
  Net before battery:   0.332 kW
  Battery ch